# Estimativa de Carga Alar e relação Tração-Peso

In [2]:
import numpy as np

def Power2Thrust_Weight(PW:    float,
                        V:     float,
                        eta_p: float = 0.8) -> float:
    """
    Converte Tracao-Peso em Potencia-Peso.
    Aeronaves a pistão utilizam o conceito de potencia em HP ao inves de tracao em lb.
    
    Args:
        PW    (float): Power-to-Weight ratio [HP/lb]
        eta_p (float): Propeller efficiency  [-]
        V     (float): Airspeed [ft/sec]
    
    Returns:
        TW    (float): Thrust-to-Weight ratio [lb/lb]
    """
    
    TW = 550*eta_p*PW/V
    
    return TW

def Thrust2Power_Weight(TW:    float,
                        V: float,
                        eta_p: float = 0.8) -> float:
    """
    Converte Tracao-Peso em Potencia-Peso.
    Aeronaves a pistão utilizam o conceito de potencia em HP ao inves de tracao em lb.
    
    Args:
        TW    (float): Thrust-to-Weight ratio [lb/lb]
        eta_p (float): Propeller efficiency  [-]
        V     (float): Airspeed [ft/sec]
    
    Returns:
        PW    (float): Power-to-Weight ratio [HP/lb]
    """
    
    PW = TW*V / (550*eta_p)
    
    return PW

## Resumo do processo
1. Estime V_max desejado <br>
       ↓
2. Calcule P/W estatístico (Tabela 5.4)<br>
       ↓
3. Estime L/D → calcule (T/W) de cruzeiro → converta para P/W<br>
       ↓
4. Ajuste o P/W de cruzeiro para condições de decolagem (Eq. 5.4)<br>
       ↓
5. Selecione o MAIOR entre os dois P/W<br>
       ↓
6. Multiplique pelo TOGW → potência do motor<br>

## Razão Tração-Peso
De acordo com a Tabela 5.2 de Raymer, nossa aeronave se aproxima da tipo de aeronave de aviação geral monomotor ("General aviation-single engine"), portanto,  é estimada uma razão Potência-Peso P/W = 0.07 hp/lb


In [3]:
P_W_inicial = 0.07  # power-to-weight ratio [horse-power/lb]

## Estimativa de razão Potência-Peso com base na Velocidade Máxima. <br>
De acordo com a Tabela 5.4, para GA-single engine, a = 0.025, C=0.22 para a equação <br>
P/W_0 = a*V_max^C <br>

Como teste, utilizou-se o P_W do Raymer para determinar a velocidade máxima da aeronave:

In [4]:
def PW_Vmax (V_max: float,
             alfa: float = 0.025,
             C: float = 0.22) -> float:
    
        """
        Tabela 5.4 do Raymer estabelece relacao estatistica de Potencia-Peso 
        com base na velocidade maxima (em kt).
        
        Args:
            alfa  (float): coef Tabel 5.4
            C     (float): coef Tabel 5.4
            V_max (float): airspeed [ft/sec]
        
        Return:
            PW0   (float): takeoff Power-to-Weight ratio [hp/lb]
        """
        # convertendo ft/sec para kt
        V_max_kt = V_max/1.688
        
        PW0 = alfa * (V_max_kt)**C
        
        return PW0

a = 0.025
C = 0.22
# Considerando a estimativa de P/W inicial, vamos verificar a velocidade maxima da aeronave
V_max_kt = (P_W_inicial/a)**(1/C) # [kt]
V_max_fts = V_max_kt * 1.688
print(V_max_fts)

181.93193908929973


Como a velocidade de cruzeiro é dada por requisito do projeto como 250 km/h (227.8 ft/s), percebeu-se que a razão Potência-Peso está subestimada. <br>
Considerando que a velocidade de cruzeiro seja 70% da velocidade máxima, conforme ocorre em outras aeronaves, divide-se a V_cruise por 0.7

In [5]:
V_cruise = 250/1.097 # [ft/s]
V_max = V_cruise/0.7 # [ft/s]

PW_V = PW_Vmax(V_max, alfa=0.025, C=0.22)
print(PW_V)

0.0795605627645534


## Thrust Matching

### Cruzeiro
Comparação do empuxo disponível durante o cruzeiro com o arrasto estimado da aeronave. <br>
(T/W)_cruise = 1 / (L/D)_cruise <br>
Utilizaremos o L/D)_cruise feito na estimativa de peso anteriormente <br>

In [13]:
def Thrust_Matching_cruise(L_D_cruise: float,
                           V_cruise: float,
                           eta_p: float = 0.8) -> float:
    """
    Equacao 5.2 do Raymer. Verificar Potencia-Peso para cruzeiro.
    L/D_cruise vem do valor calculado na Entrega 1 do trabalho
    
    Args:
        L_D_cruise (float): Lift-to-Drag ratio           [lb/lb]
        eta_p      (float): Propeller efficiency         [-]
        V_cruise   (float): airspeed in cruise           [ft/sec]
    
    Return:
        TW_cruise  (float): cruise Thrust-to-Weight ratio [lb/lb]
        PW_cruise  (float): cruise Power-to-Weight ratio [hp/lb]
    """
    
    TW_cruise = 1 / (L_D_cruise)
    PW_cruise = Thrust2Power_Weight(TW_cruise, V_cruise, eta_p)
    
    return TW_cruise, PW_cruise

# V_cruise = 250/1.097 # [ft/s]

TW_cruise, PW_cruise = Thrust_Matching_cruise(L_D_cruise = 18.6, V_cruise=250/1.097, eta_p=0.8)

print("TW_cruise: ", TW_cruise, "lb/lb")
print("PW_cruise: ", PW_cruise, "hp/lb")


TW_cruise:  0.05376344086021505 lb/lb
PW_cruise:  0.027846316845640512 hp/lb


### Climb

Tabela F.2, Anexo F, V_vertical:
Para uso militar FAR MIL-C5011A: 500 fpm
Para uso civil (Config adotada): 300 fpm

*_DÚVIDA:_ Como saber o L/D)_climb e V_Vertical?*

In [ ]:
T_W_climb = 1 / L_D_climb + V_vertical / V

## Relações de T/W em condições de decolagem

Equação 5.4

In [16]:
def Thrust_Matching_takeoff(PW_cruise: float,
                            TW_cruise: float,
                            W1W0: float = 0.97,
                            W2W1: float = 0.985,
                            Tcruise_Tto: float = 0.70) -> float:
    """
    Calculou-se a tracao em cruzeiro. Necessita-se da tracao na decolagem.
    
    Args:
        PW_cruise   (float): Calculado no Thrust_Matching [hp/lb]
        TW_cruise   (float): Calculado no Thrust_Matching [lb/lb]
        W1W0        (float): #razao da primeira etapa da missao (decolagem), Raymer define 0.97
        W2W1        (float): #razao da segunda etapa da missao (climb), Raymer define 0.985
        Tto_Tcruise (float): #razao tracao de decolagem e tracao de cruzeiro, valores variam de
                              [0.60-0.80], se possivel, verificar se tem dados dos fabricantes
    Return:
        TW_takeoff  (float):  takeoff Thrust-to-Weight ratio [lb/lb]
        PW_takeoff  (float):  takeoff Power-to-Weight ratio  [hp/lb]
    """
    Wcruise_Wtakeoff = W1W0*W2W1
    
    TW_takeoff = TW_cruise * Wcruise_Wtakeoff / Tcruise_Tto
    PW_takeoff = PW_cruise * Wcruise_Wtakeoff / Tcruise_Tto
    
    return TW_takeoff, PW_takeoff

TW_takeoff, PW_takeoff = Thrust_Matching_takeoff(PW_cruise=PW_cruise, TW_cruise=TW_cruise)
print("TW_takeoff", TW_takeoff, "lb/lb")
print("PW_takeoff", PW_takeoff, "hp/lb")

TW_takeoff 0.0733832565284178 lb/lb
PW_takeoff 0.038008233471667464 hp/lb


Fazemos a seleção com base no maior valor de P/W:

In [18]:
list_PW = [P_W_inicial, PW_cruise, PW_takeoff, PW_V]
PW = max(list_PW)
print(f"Valor selecionado de P/W: {PW:.5f} hp/lb")

Valor selecionado de P/W: 0.07956 hp/lb


Dessa forma, seleciona-se o requisito mínimo de potência do motor, multipolicando P/W pelo peso de decolagem (MTOW, ou seja, W0):

In [ ]:
def motorPower(PW: float,
               W0: float)-> float:
    """
    Calcula potencia do motor.
    
    Args:
        PW      (float): higher power-to-weight ratio [hp/lb]
        W0      (float): MTOW                         [lb]
        
    Return:
        P_motor (float): motor power                  [hp]
    """
    P_motor = PW*W0
    
    return P_motor

Power_motor = motorPower(PW=PW, W0=523)
print(f"A Potência do motor será de {Power_motor:.2f} HP")

A Potência do motor será de 41.61 HP


# Análise de Restrições Gudmundsson

## T/W para curva nivelada com velocidade constante

Por enquanto, não será feita, restrição não existe no nosso projeto

## T/W para energia específica desejada

Por enquanto, não será feita, restrição não existe no nosso projeto

## T/W para uma razão de subida desejada

Não é um requisito do nosso projeto, mas é interessante avaliar.

In [ ]:
TW_ROC = V_v/V_climb + q/(WS)*CD_min + k/q